# 02. defaultdict

**Tier:** Intermediate. **Why you'd use this:** You group words by first letter into lists. The first time you see a letter, you need an empty list ready. `defaultdict` builds that missing value for you.

**After this lesson you can:** group with `defaultdict(list)`; explain `__missing__`; convert back to a plain dict when you need to.
About 25 minutes. **Uses:** [Counter](01-counter-lesson.ipynb), [Dicts](../04-collections/03-dicts-lesson.ipynb)


## Quick review

1. Why does `Counter` return `0` for a missing key?
2. Is `Counter` a dict subclass?

<details><summary>Answers</summary>

So tallies can increment on first sight. Yes.
</details>


## The idea

`defaultdict` is a dict subclass. You pass a zero-argument factory such as `list` or `int`. On a missing key, it calls that factory, stores the result, and returns it. That path is `__missing__`. Grouping becomes append-only with no `KeyError` guard.

<details><summary>Explain more</summary>

`groups[key].append(value)` works on the first sighting because the missing key creates `[]` first. Call `dict(groups)` when an API wants a plain dict.
</details>


## See it work

Group words by their first letter.


In [1]:
from collections import defaultdict


def by_initial(words):
    # 1. Missing keys start as empty lists.
    groups = defaultdict(list)
    for word in words:
        groups[word[0]].append(word)
    return dict(groups)


print(by_initial(["apple", "ant", "bee"]))

{'a': ['apple', 'ant'], 'b': ['bee']}


## Check yourself

Answer these before you run the cell.

1. After `groups = defaultdict(list)`, what is `groups["z"]` before any append?
2. Does reading that missing key leave `"z"` stored?


In [2]:
groups = defaultdict(list)
print(groups["z"])
print(dict(groups))

[]
{'z': []}


<details><summary>Answers</summary>

It is `[]`. Yes. Reading a missing key inserts the factory result, so `"z"` stays in the mapping.
</details>


## Why it works this way

Ordinary dict lookup raises when a key is absent. `defaultdict` overrides that path through `__missing__`. The factory runs with no arguments, the result is stored under the key, and that result is returned. Later lookups hit the stored value like any dict. That is why grouping code can append on first sight without an `if key not in groups` guard. Deep dive: [How dict and set work](../../internals/04-dicts/01-hash-table-lesson.ipynb).

The factory choice is the type of default. `list` builds groups. `int` builds counters that start at `0`. `set` builds membership bags. Passing a ready list as the factory is wrong: the factory must be callable with no arguments. `defaultdict(list)` passes the `list` type. `defaultdict([])` fails because a list is not callable. A custom zero-argument function works when the default needs setup beyond a type call.

Cost matches a dict plus one factory call per first sighting of a key. Memory holds one entry per key you touched, including keys you only read. That last point is the design trade-off: convenience on write paths can insert keys on read paths. A membership test that indexes the mapping will create empty groups you never meant to keep. Convert with `dict(...)` when you want a snapshot that no longer auto-fills, or when an API rejects `defaultdict` instances.

`Counter` is the specialized zero-default for integers with ranking helpers. Prefer `Counter` for frequencies. Prefer `defaultdict` when the default is a container you will mutate. Prefer a plain dict when a missing key should raise and stop you.


In [3]:
groups = defaultdict(list)
groups["a"].append("ant")
print(groups["a"])
print("z" in groups)
print(groups["z"])
print("z" in groups)
print(dict(groups))

['ant']
False
[]
True
{'a': ['ant'], 'z': []}


## When to use it

Use `defaultdict` to group or accumulate into containers. Use `Counter` for pure counts. Use a plain dict when a missing key should raise. Don't pass a list or dict instance as the factory. Pass the type or a zero-argument function.


## Common mistake

Reading a key to "check" it, which inserts an empty default and pollutes the mapping.


In [4]:
groups = defaultdict(list)
if groups["z"]:
    print("has z")
print(dict(groups))

{'z': []}


Test with `key in groups` when you do not want insertion. Only index when you intend to use the default.


In [5]:
groups = defaultdict(list)
if "z" in groups:
    print("has z")
print(dict(groups))

{}


## Try it

Change `by_initial` to group by word length instead of first letter. Which part of the key expression changes, and do you still need `defaultdict(list)`?


## Recap

- `defaultdict(factory)` calls `factory` on a missing key through `__missing__`.
- A read can insert. Use `in` when you only want to test.
- Convert with `dict(...)` when you need a plain mapping.

**Next:** [practice](02-defaultdict-practice.ipynb)
